# Agents for HPC Operations

### Job efficiency and queue time on a Slurm cluster

Today we will build a working agent from four pieces: a tool, a loop, a description of
the data, and notes that explain what the data means.

The data is four weeks of Slurm accounting from a dummy national platform, **Kōwhai**:
87,439 jobs from 201 researchers across 18 projects, 18.9 million core-hours charged
on 314 nodes, plus a scheduler sample taken every 15 minutes.

You need to have used Slurm as a user (`sbatch`, `squeue`, walltime limits) and to
read basic SQL and Python. You do not need any machine learning background: the model
is a service we call, and every piece we build around it is ordinary code.

## What is in this notebook

Run the cells in order.

| Part | Title | What you learn |
|---|---|---|
| 1 | One model response | Make a request and inspect the returned message |
| 2 | The accounting data | Check coverage and grain before trusting an aggregate |
| 3 | One tool call by hand | Follow the request, execution and result cycle |
| 4 | Automate the loop | Repeat the cycle and combine two tools |
| 5 | From narrow tools to SQL | Replace question-specific tools with SQL, then guide it with a schema card |
| 6 | When is the cluster busy? | See how missing time-zone context changes an answer |
| 7 | Three numbers for one wait | Separate held, queued and estimated time, and find out why the estimate is wrong |
| 8 | Guardrails | Cap rows, require time filters and look up stored values |
| 9 | The second table | Write a schema card and combine two data sources |
| 10 | Investigate an incident | Use the completed agent and cross-check its answer |
| 11 | A job worth handing over | See the task that actually justifies the machinery |
| 12 | Measure it before you trust it | Variance, cost and the error that hides, then decide agent or query |

The last cell, `_selfcheck()`, runs every tool without calling a model. If something
fails, run it to find out whether the problem is in the setup and tools or in the
model call.

## 0. Setup

Run the install cell once. The next cell reads `OPENROUTER_API_KEY` from the
environment or a local `.env` file. If neither exists, it asks for the key without
displaying what you type.

There is nothing to download. The dataset is generated locally in about five seconds
and written to two Parquet files.

OpenRouter is a broker: one API key, many models, all behind the same
OpenAI-compatible interface. Nothing here depends on this particular model; change
`MODEL` to try another. A complete run of this notebook makes a few dozen model calls
and costs a few cents.

In [ ]:
# Pinned so the workshop behaves the same everywhere. Safe to re-run.
!pip install -q duckdb==1.5.5 matplotlib openai==2.53.0 pandas pyarrow tabulate==0.10.0

In [ ]:
import json
import os
import re
from collections.abc import Callable, Mapping
from getpass import getpass

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from openai import OpenAI

# Any model that supports tool calling works here. This one is fast and cheap.
MODEL = "google/gemini-3.5-flash-lite"


def _load_key() -> str:
    if os.environ.get("OPENROUTER_API_KEY"):
        return os.environ["OPENROUTER_API_KEY"]
    if os.path.exists(".env"):
        for line in open(".env"):
            if line.startswith("OPENROUTER_API_KEY="):
                return line.split("=", 1)[1].strip().strip("\"'")
    return getpass("OPENROUTER_API_KEY: ")


client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=_load_key())

### Where the data comes from

Two tables, both standing in for things you already have on a real cluster.

| Table | Real source | Grain |
|---|---|---|
| `jobs` | `sacct` | one row per job allocation |
| `sched_15m` | `sinfo` and `squeue`, sampled | one row per 15 minutes, per partition |

The last section of the notebook gives the `sacct` command that produces the same
columns, and notes where a real export will differ.

The next cell is the generator. It is plumbing, not content: it is hidden by default
and you can run it without reading it.

In [ ]:
#@title Build the synthetic dataset  (click the arrow to read the code)
"""Synthetic Slurm accounting data for the Kowhai workshop.

Stands in for `sacct` and `sinfo` on a real cluster. You do not need to read this
to do the workshop; skip to Part 1 once it has run.
"""
import os

import duckdb
import numpy as np
import pandas as pd

SEED = 62
UTC_START = np.datetime64("2026-07-05T12:00")   # Mon 6 Jul 00:00 NZST
DAYS = 28
MINUTES = DAYS * 24 * 60
TZ = 12 * 60  # NZST minutes ahead of UTC

PARTITIONS = {
    "debug":   dict(nodes=4,   cpn=128, mem=480_000,   gpus=0, max_min=15),
    "large":   dict(nodes=240, cpn=128, mem=480_000,   gpus=0, max_min=4320),
    "long":    dict(nodes=40,  cpn=128, mem=480_000,   gpus=0, max_min=20160),
    "hugemem": dict(nodes=6,   cpn=128, mem=4_030_000, gpus=0, max_min=10080),
    "gpu":     dict(nodes=24,  cpn=64,  mem=480_000,   gpus=4, max_min=1440),
}

PROJECTS = [
    ("uoa03521", "Te Whare Wānanga o Tāmaki Makaurau — Molecular Dynamics", "University of Auckland", "Chemistry"),
    ("uoa04412", "Te Whare Wānanga o Tāmaki Makaurau — Kauri Dieback Metagenomics", "University of Auckland", "Biology"),
    ("uow02918", "Te Whare Wānanga o Waikato — Coastal Ocean Modelling", "University of Waikato", "Earth Sciences"),
    ("vuw03102", "Te Herenga Waka — Seismic Wave Propagation", "Victoria University of Wellington", "Earth Sciences"),
    ("uoo02755", "Te Whare Wānanga o Ōtākou — Genomic Epidemiology", "University of Otago", "Biology"),
    ("cawt01130", "Cawthron Institute — Harmful Algal Bloom Forecasting", "Cawthron Institute", "Biology"),
    ("niwa02901", "NIWA — Regional Climate Downscaling", "NIWA", "Earth Sciences"),
    ("niwa03340", "NIWA — Antarctic Sea Ice Reanalysis", "NIWA", "Earth Sciences"),
    ("mwlr01862", "Manaaki Whenua — Land Cover Classification", "Manaaki Whenua Landcare Research", "Earth Sciences"),
    ("plnt02211", "Plant & Food Research — Kiwifruit Pangenome", "Plant and Food Research", "Biology"),
    ("agr01204",  "AgResearch — Ruminant Methane Modelling", "AgResearch", "Biology"),
    ("uoc02640", "Te Whare Wānanga o Waitaha — Computational Fluid Dynamics", "University of Canterbury", "Engineering"),
    ("uoc03881", "Te Whare Wānanga o Waitaha — Gravitational Wave Search", "University of Canterbury", "Physics"),
    ("mas02409", "Te Kunenga ki Pūrehuroa — Protein Structure Prediction", "Massey University", "Biology"),
    ("aut01527", "Te Wānanga Aronui o Tāmaki Makau Rau — Neural Speech Models", "Auckland University of Technology", "Computer Science"),
    ("gns02066", "GNS Science — Geothermal Reservoir Simulation", "GNS Science", "Earth Sciences"),
    ("lcr01998", "Te Pūnaha Matatini — Epidemic Network Models", "Te Pūnaha Matatini", "Mathematics"),
    ("mfe00812", "Ministry for the Environment — Freshwater Quality Modelling", "Ministry for the Environment", "Earth Sciences"),
]

FIRSTS = ["hana", "rewi", "mere", "tane", "aroha", "kiri", "manaia", "ngaio", "rangi", "tui",
          "james", "sarah", "wei", "priya", "chen", "aditi", "olivia", "liam", "noah", "emma",
          "hemi", "ana", "raj", "yuki", "sofia", "ben", "grace", "leo", "maia", "finn"]
LASTS = ["kerekere", "ngata", "clifton", "harrison", "zhang", "patel", "wilson", "obrien",
         "tahana", "murray", "singh", "nakamura", "lopez", "brown", "walker", "cheng",
         "waititi", "reid", "fraser", "kumar"]


def local_hour(ts_utc):
    """UTC_START is exactly Monday 00:00 NZST, so offsets are measured from it."""
    m = (ts_utc - UTC_START) / np.timedelta64(1, "m")
    return (m / 60.0) % 24.0


def local_dow(ts_utc):
    # 0 = Monday local
    m = (ts_utc - UTC_START) / np.timedelta64(1, "m")
    return ((m // (24 * 60)).astype(int)) % 7


# Submissions per local hour, relative to the 11:00 peak. A working day, not a curve.
HOURLY = np.array([
    0.11, 0.08, 0.05, 0.04, 0.04, 0.06, 0.11, 0.22,   # 00-07
    0.46, 0.80, 0.95, 1.00, 0.84, 0.90, 0.95, 0.90,   # 08-15
    0.84, 0.68, 0.50, 0.40, 0.34, 0.28, 0.21, 0.15,   # 16-23
])


def submit_intensity(ts):
    h = local_hour(ts).astype(int) % 24
    d = local_dow(ts)
    week = np.where(d >= 5, 0.45, 1.0)
    return np.clip(HOURLY[h] * week, 0.02, 1.0)


def sample_submits(n, rng, spread=1.0):
    """Diurnal + weekly submission pattern in local time, returned as UTC."""
    out = []
    while sum(len(o) for o in out) < n:
        cand = UTC_START + (rng.random(n * 3) * MINUTES).astype("timedelta64[m]")
        keep = cand[rng.random(cand.size) < submit_intensity(cand) ** spread]
        out.append(keep)
    return np.sort(np.concatenate(out)[:n])


def cluster_load(ts, partition, rng):
    """0-1 pressure on the queue at this moment."""
    h = local_hour(ts)
    d = local_dow(ts)
    base = 0.55 + 0.32 * np.exp(-0.5 * ((h - 13.0) / 5.0) ** 2)
    base = np.where(d >= 5, base - 0.16, base)
    # the incident: Sat 25 - Mon 27 July NZST, `large` saturated
    inc0 = np.datetime64("2026-07-24T12:00")   # Sat 25 Jul 00:00 NZST
    inc1 = np.datetime64("2026-07-27T06:00")   # Mon 27 Jul 18:00 NZST
    if partition in ("large", "debug"):
        base = np.where((ts >= inc0) & (ts < inc1), np.minimum(base + 0.42, 1.10), base)
    # maintenance reservation: Tue 21 Jul 08:00-14:00 NZST drains 40 large nodes
    m0 = np.datetime64("2026-07-20T20:00")
    m1 = np.datetime64("2026-07-21T02:00")
    if partition == "large":
        base = np.where((ts >= m0) & (ts < m1), base + 0.18, base)
    return np.clip(base + rng.normal(0, 0.06, ts.size), 0.05, 1.15)


def wait_minutes(ts, partition, req_nodes, qos, fairshare, rng):
    load = cluster_load(ts, partition, rng)
    size = 1.0 + np.log1p(req_nodes) * 1.35
    pressure = np.exp(4.6 * (load - 0.55))
    base = {"debug": 0.6, "large": 4.0, "long": 18.0, "hugemem": 26.0, "gpu": 14.0}[partition]
    mu = base * size * pressure * fairshare
    w = rng.lognormal(np.log(np.maximum(mu, 0.3)), 0.85)
    w = np.where(qos == "debug", w * 0.08, w)
    return np.clip(w, 0.05, 60 * 30)


def make_block(rng, n, partition, name_pool, spread, req_nodes, cpus_per_node_used,
               mem_frac, timelimit, walltime_use, cpu_eff, mem_eff, gpus=0,
               gpu_util=None, dep_mean=0.0, proj_idx=None, user_idx=None):
    p = PARTITIONS[partition]
    submit = sample_submits(n, rng, spread)
    req_nodes = np.asarray(req_nodes)
    req_cpus = req_nodes * cpus_per_node_used
    req_mem = np.round(p["mem"] * mem_frac * req_nodes / 1000.0) * 1000.0

    tl = np.clip(np.round(timelimit), 5, p["max_min"])
    elapsed = np.clip(np.round(tl * walltime_use), 1, tl)

    state = rng.choice(
        ["COMPLETED", "FAILED", "TIMEOUT", "CANCELLED", "OUT_OF_MEMORY", "NODE_FAIL"],
        size=n, p=[0.855, 0.070, 0.032, 0.030, 0.010, 0.003])
    elapsed = np.where(state == "TIMEOUT", tl, elapsed)
    elapsed = np.where(state == "FAILED", np.maximum(1, np.round(elapsed * rng.beta(1.2, 3.0, n))), elapsed)
    elapsed = np.where(state == "OUT_OF_MEMORY", np.maximum(1, np.round(elapsed * rng.beta(1.5, 3.0, n))), elapsed)
    elapsed = np.where(state == "NODE_FAIL", np.maximum(1, np.round(elapsed * rng.beta(1.2, 2.0, n))), elapsed)
    cancelled_pending = (state == "CANCELLED") & (rng.random(n) < 0.35)
    elapsed = np.where((state == "CANCELLED") & ~cancelled_pending,
                       np.maximum(1, np.round(elapsed * rng.beta(1.0, 4.0, n))), elapsed)

    eff = np.clip(cpu_eff, 0.002, 1.0)
    total_cpu = np.round(req_cpus * elapsed * eff, 1)
    rss = np.round(req_mem * np.clip(mem_eff, 0.01, 1.35))
    rss = np.where(state == "OUT_OF_MEMORY", np.round(req_mem * rng.uniform(0.97, 1.0, n)), rss)
    rss = np.minimum(rss, req_mem * 1.02)

    fairshare = rng.uniform(0.6, 1.9, n)
    qos = np.where(np.full(n, partition == "debug"), "debug", "normal")
    dep = np.where(rng.random(n) < (0.85 if dep_mean > 0 else 0.0),
                   rng.exponential(max(dep_mean, 1e-6), n), 0.0)
    dep = np.where(rng.random(n) < 0.03, dep + rng.uniform(600, 4300, n), dep)
    eligible = submit + np.round(dep).astype("timedelta64[m]")

    planned = wait_minutes(eligible, partition, req_nodes, qos, fairshare, rng)
    start = eligible + np.round(planned).astype("timedelta64[m]")
    end = start + elapsed.astype("timedelta64[m]")

    df = pd.DataFrame(dict(
        job_name=rng.choice(name_pool, n),
        partition=partition, qos=qos,
        submit_ts=submit, eligible_ts=eligible, start_ts=start, end_ts=end,
        state=state, req_nodes=req_nodes, req_cpus=req_cpus,
        req_mem_mb=req_mem.astype(np.int64), req_gpus=req_nodes * gpus,
        timelimit_min=tl.astype(np.int64), elapsed_min=elapsed.astype(np.int64),
        planned_min=np.round(planned, 1), total_cpu_min=total_cpu,
        max_rss_mb=rss.astype(np.int64),
        gpu_util_pct=(np.round(gpu_util, 1) if gpu_util is not None else np.nan),
        _cancelled_pending=cancelled_pending,
        _proj=proj_idx if proj_idx is not None else rng.integers(0, len(PROJECTS), n),
        _user=user_idx if user_idx is not None else rng.integers(0, 340, n),
    ))
    return df


def build(rng):
    blocks = []

    # 1. Nextflow / Snakemake pipeline tasks - dependency held, small, efficient
    n = 46000
    blocks.append(make_block(
        rng, n, "large",
        ["nf-FASTQC", "nf-BWAMEM", "nf-MARKDUP", "nf-HAPLOTYPECALLER", "snakemake-align",
         "nf-MULTIQC", "snakemake-count"],
        spread=0.7,
        req_nodes=np.ones(n, int), cpus_per_node_used=rng.choice([2, 4, 8, 16], n, p=[.3, .35, .25, .1]),
        mem_frac=rng.uniform(0.03, 0.18, n),
        timelimit=rng.choice([60, 120, 240, 480], n, p=[.35, .35, .2, .1]),
        walltime_use=rng.beta(1.6, 3.2, n),
        cpu_eff=rng.beta(7, 2.4, n),
        mem_eff=rng.beta(2.2, 3.0, n),
        dep_mean=95.0))

    # 2. MPI simulation - multi node, well tuned
    n = 2200
    nodes = rng.choice([1, 2, 4, 8, 16, 32], n, p=[.34, .28, .19, .11, .06, .02])
    blocks.append(make_block(
        rng, n, "large",
        ["gromacs_prod", "wrf_nested", "openfoam_les", "specfem3d", "nemo_ocean", "vasp_relax"],
        spread=1.0,
        req_nodes=nodes, cpus_per_node_used=128,
        mem_frac=rng.uniform(0.25, 0.75, n),
        timelimit=rng.choice([360, 720, 1440, 2880], n, p=[.3, .35, .25, .1]),
        walltime_use=rng.beta(2.6, 2.0, n),
        cpu_eff=rng.beta(14, 1.9, n),
        mem_eff=rng.beta(3.0, 2.4, n)))

    # 3. Serial R / Python asking for a whole node - the waste
    n = 7200
    cpus = rng.choice([32, 64, 128], n, p=[.4, .3, .3])
    blocks.append(make_block(
        rng, n, "large",
        ["run_model.R", "analysis.R", "bootstrap.R", "process.py", "fit_glmm.R"],
        spread=0.9,
        req_nodes=np.ones(n, int), cpus_per_node_used=cpus,
        mem_frac=rng.uniform(0.3, 0.95, n),
        timelimit=rng.choice([240, 480, 1440, 2880], n, p=[.25, .3, .3, .15]),
        walltime_use=rng.beta(1.3, 4.0, n),
        cpu_eff=rng.uniform(0.9, 1.7, n) / cpus,   # single threaded on a whole node
        mem_eff=rng.beta(1.3, 6.0, n)))

    # 4. GPU training - low CPU efficiency by design
    n = 8600
    gu = np.clip(rng.beta(4.5, 2.0, n) * 100, 3, 99)
    blocks.append(make_block(
        rng, n, "gpu",
        ["train_asr", "finetune_llm", "alphafold_gpu", "torch_ddp", "cryosparc_gpu"],
        spread=0.9,
        req_nodes=np.ones(n, int), cpus_per_node_used=rng.choice([4, 8, 16], n, p=[.3, .5, .2]),
        mem_frac=rng.uniform(0.1, 0.5, n),
        timelimit=rng.choice([240, 480, 720, 1440], n, p=[.2, .3, .3, .2]),
        walltime_use=rng.beta(2.0, 2.4, n),
        cpu_eff=rng.beta(1.6, 9.0, n),
        mem_eff=rng.beta(2.0, 3.0, n),
        gpus=rng.choice([1, 2, 4], n, p=[.6, .25, .15]), gpu_util=gu))

    # 5. Interactive / MATLAB - huge walltime over-request
    n = 6400
    blocks.append(make_block(
        rng, n, "large",
        ["matlab_session", "jupyter", "sinteractive", "ondemand_rstudio"],
        spread=1.3,
        req_nodes=np.ones(n, int), cpus_per_node_used=rng.choice([4, 8, 16, 32], n, p=[.3, .3, .25, .15]),
        mem_frac=rng.uniform(0.05, 0.4, n),
        timelimit=rng.choice([480, 1440, 2880], n, p=[.4, .4, .2]),
        walltime_use=rng.beta(1.0, 9.0, n),
        cpu_eff=rng.beta(1.4, 7.0, n),
        mem_eff=rng.beta(1.5, 5.0, n)))

    # 6. hugemem assembly
    n = 140
    blocks.append(make_block(
        rng, n, "hugemem",
        ["hifiasm", "spades_meta", "trinity_asm", "canu_correct"],
        spread=1.0,
        req_nodes=np.ones(n, int), cpus_per_node_used=rng.choice([64, 128], n, p=[.4, .6]),
        mem_frac=rng.uniform(0.35, 0.95, n),
        timelimit=rng.choice([1440, 2880, 5760, 10080], n, p=[.3, .3, .3, .1]),
        walltime_use=rng.beta(2.0, 2.5, n),
        cpu_eff=rng.beta(4.0, 3.0, n),
        mem_eff=rng.beta(3.0, 2.2, n)))

    # 7. long partition - climate runs
    n = 180
    nodes = rng.choice([1, 2, 4, 8], n, p=[.45, .3, .18, .07])
    blocks.append(make_block(
        rng, n, "long",
        ["cesm_hist", "roms_hindcast", "mom6_spinup", "chem_transport"],
        spread=1.0,
        req_nodes=nodes, cpus_per_node_used=128,
        mem_frac=rng.uniform(0.2, 0.6, n),
        timelimit=rng.choice([4320, 10080, 20160], n, p=[.4, .4, .2]),
        walltime_use=rng.beta(3.0, 2.0, n),
        cpu_eff=rng.beta(9.0, 2.2, n),
        mem_eff=rng.beta(2.6, 2.6, n)))

    # 8. debug
    n = 11000
    blocks.append(make_block(
        rng, n, "debug",
        ["test", "hello_mpi", "check_env", "quicktest", "debug_run"],
        spread=1.1,
        req_nodes=np.ones(n, int), cpus_per_node_used=rng.choice([1, 2, 4, 8], n, p=[.4, .3, .2, .1]),
        mem_frac=rng.uniform(0.02, 0.2, n),
        timelimit=np.full(n, 15),
        walltime_use=rng.beta(1.1, 5.0, n),
        cpu_eff=rng.beta(2.2, 2.6, n),
        mem_eff=rng.beta(1.5, 5.0, n)))

    # --- planted incident: 3,200 array tasks, 128 cpus each, single threaded ---
    n = 6000
    inc_submit = (np.datetime64("2026-07-24T12:00")
                  + np.round(rng.exponential(95, n)).astype("timedelta64[m]"))
    elapsed = np.round(rng.normal(64, 11, n)).clip(18, 140)
    inc = pd.DataFrame(dict(
        job_name="kauri_bin_annotate",
        partition="large", qos="normal",
        submit_ts=inc_submit, eligible_ts=inc_submit,
        state="COMPLETED", req_nodes=1, req_cpus=128, req_mem_mb=480000, req_gpus=0,
        timelimit_min=1440, elapsed_min=elapsed.astype(np.int64),
        total_cpu_min=np.round(elapsed * rng.uniform(0.95, 1.25, n), 1),
        max_rss_mb=np.round(rng.uniform(9000, 26000, n)).astype(np.int64),
        gpu_util_pct=np.nan,
        _cancelled_pending=False,
        _proj=1,          # uoa04412 Kauri Dieback Metagenomics
        _user=7,
    ))
    # they trickle through the queue over ~2.5 days
    order = np.argsort(rng.random(n))
    slot = np.zeros(n)
    slot[order] = np.linspace(20, 3600, n) + rng.normal(0, 90, n)
    inc["planned_min"] = np.round(np.clip(slot, 5, None), 1)
    inc["start_ts"] = inc["eligible_ts"] + pd.to_timedelta(inc["planned_min"].round(), unit="m")
    inc["end_ts"] = inc["start_ts"] + pd.to_timedelta(inc["elapsed_min"], unit="m")
    blocks.append(inc)

    # --- second, quieter anomaly: same job resubmitted, always TIMEOUT ---
    n = 41
    sub = (np.datetime64("2026-07-06T00:00")
           + np.round(np.linspace(0, 26 * 1440, n) + rng.normal(0, 120, n)).astype("timedelta64[m]"))
    tmo = pd.DataFrame(dict(
        job_name="seismic_inv_full", partition="large", qos="normal",
        submit_ts=sub, eligible_ts=sub, state="TIMEOUT",
        req_nodes=2, req_cpus=256, req_mem_mb=960000, req_gpus=0,
        timelimit_min=4320, elapsed_min=4320,
        total_cpu_min=np.round(256 * 4320 * rng.uniform(0.80, 0.93, n), 1),
        max_rss_mb=np.round(rng.uniform(180000, 320000, n)).astype(np.int64),
        gpu_util_pct=np.nan, _cancelled_pending=False, _proj=3, _user=2,
        planned_min=np.round(rng.lognormal(np.log(140), 0.8, n), 1),
    ))
    tmo["start_ts"] = tmo["eligible_ts"] + pd.to_timedelta(tmo["planned_min"].round(), unit="m")
    tmo["end_ts"] = tmo["start_ts"] + pd.to_timedelta(tmo["elapsed_min"], unit="m")
    blocks.append(tmo)

    df = pd.concat(blocks, ignore_index=True)
    df = df[df["submit_ts"] < np.datetime64("2026-08-02T12:00")].copy()
    df = df.sort_values("submit_ts").reset_index(drop=True)

    # identity columns
    proj = np.array(PROJECTS, dtype=object)
    df["account"] = proj[df["_proj"].values, 0]
    df["project_name"] = proj[df["_proj"].values, 1]
    df["institution"] = proj[df["_proj"].values, 2]
    users = np.array([f"{FIRSTS[i % len(FIRSTS)][0]}{LASTS[(i * 7) % len(LASTS)]}{'' if i < 20 else i % 90}"
                      for i in range(340)])
    df["user"] = users[df["_user"].values % 340]
    df.loc[df["_proj"] == 1, "user"] = np.where(
        rng.random((df["_proj"] == 1).sum()) < 0.55, "hkerekere",
        users[rng.integers(0, 340, (df["_proj"] == 1).sum())])
    df.loc[(df["_proj"] == 1) & (df["job_name"] == "kauri_bin_annotate"), "user"] = "hkerekere"
    df.loc[df["job_name"] == "seismic_inv_full", "user"] = "rclifton"

    df["job_id"] = np.arange(4_180_000, 4_180_000 + len(df))
    is_array = df["job_name"].str.startswith(("nf-", "snakemake")).values
    seq = np.cumsum(is_array) - 1
    df["array_task_id"] = np.where(is_array, seq % 250, -1)
    df["array_job_id"] = np.where(is_array, 5_000_000 + seq // 250, -1)
    kauri = (df["job_name"] == "kauri_bin_annotate").values
    df.loc[kauri, "array_job_id"] = 5_900_017
    df.loc[kauri, "array_task_id"] = np.arange(kauri.sum())

    # cancelled-while-pending jobs never start
    cp = df["_cancelled_pending"].fillna(False).values.astype(bool)
    for c in ["start_ts", "end_ts"]:
        df[c] = df[c].astype("datetime64[ns]")
        df.loc[cp, c] = pd.NaT
    df.loc[cp, ["elapsed_min", "total_cpu_min", "max_rss_mb", "planned_min"]] = np.nan

    # Slurm's estimated start time, produced by the backfill scheduler.
    # Pessimistic, because it assumes every running job uses its full time limit.
    nn = len(df)
    factor = rng.lognormal(np.log(3.1), 0.62, nn)
    factor = np.where(rng.random(nn) < 0.09, rng.uniform(0.35, 0.9, nn), factor)
    est = df["eligible_ts"].values + pd.to_timedelta(
        np.round(df["planned_min"].fillna(0).values * factor), unit="m")
    has_est = (df["planned_min"].fillna(0) > 4) & (rng.random(nn) < 0.72) & ~cp
    df["est_start_ts"] = np.where(has_est, est, np.datetime64("NaT"))

    reason = np.where(df["planned_min"].fillna(0) > 4, "Priority", None)
    big = df["req_nodes"] >= 8
    reason = np.where(big & (df["planned_min"].fillna(0) > 4), "Resources", reason)
    dep_held = (df["eligible_ts"] - df["submit_ts"]) > pd.Timedelta(minutes=2)
    reason = np.where(dep_held, "Dependency", reason)
    maint = (df["start_ts"] >= np.datetime64("2026-07-20T20:00")) & \
            (df["start_ts"] < np.datetime64("2026-07-21T04:00")) & (df["partition"] == "large")
    reason = np.where(maint & (df["planned_min"].fillna(0) > 30),
                      "ReqNodeNotAvail, Reserved for maintenance", reason)
    df["last_reason"] = reason
    df.loc[dep_held, "est_start_ts"] = pd.NaT  # no estimate while a dependency is unmet

    df["exit_code"] = np.where(df["state"] == "COMPLETED", "0:0",
                               np.where(df["state"] == "OUT_OF_MEMORY", "0:125",
                                        np.where(df["state"] == "TIMEOUT", "0:15", "1:0")))
    # sacct export taken at the end of the window: jobs still running are not in it
    df = df[df["end_ts"].notna() & (df["end_ts"] < np.datetime64("2026-08-02T12:00"))
            | df["_cancelled_pending"].fillna(False)].copy()

    for c in ["submit_ts", "eligible_ts", "est_start_ts", "start_ts", "end_ts"]:
        df[c] = df[c].astype("datetime64[us]")
    for c in ["elapsed_min", "max_rss_mb", "timelimit_min", "req_cpus", "req_nodes"]:
        df[c] = df[c].astype("Int64")

    cols = ["job_id", "array_job_id", "array_task_id", "job_name", "user", "account", "project_name",
            "institution", "partition", "qos", "state", "exit_code",
            "submit_ts", "eligible_ts", "est_start_ts", "start_ts", "end_ts",
            "timelimit_min", "elapsed_min", "planned_min",
            "req_nodes", "req_cpus", "req_mem_mb", "req_gpus",
            "total_cpu_min", "max_rss_mb", "gpu_util_pct", "last_reason"]
    return df[cols]


def build_sched(jobs, rng):
    con = duckdb.connect()
    con.register("j", jobs)
    parts = pd.DataFrame([dict(partition=k, nodes_total=v["nodes"],
                               cpus_total=v["nodes"] * v["cpn"]) for k, v in PARTITIONS.items()])
    con.register("p", parts)
    sql = """
    WITH b AS (
      SELECT UNNEST(generate_series(TIMESTAMP '2026-07-05 12:00',
                                    TIMESTAMP '2026-08-02 11:45',
                                    INTERVAL 15 MINUTE)) AS ts
    ),
    grid AS (SELECT b.ts, p.partition, p.nodes_total, p.cpus_total FROM b CROSS JOIN p),
    run AS (
      SELECT g.ts, g.partition,
             COUNT(*) AS jobs_running,
             SUM(j.req_nodes) AS nodes_alloc,
             SUM(j.req_cpus) AS cpus_alloc
      FROM grid g JOIN j ON j.partition = g.partition
        AND j.start_ts <= g.ts AND j.end_ts > g.ts
      GROUP BY 1, 2
    ),
    pend AS (
      SELECT g.ts, g.partition,
             COUNT(*) AS jobs_pending,
             SUM(j.req_cpus) AS cpus_pending_requested,
             MAX(date_diff('minute', j.eligible_ts, g.ts)) AS oldest_pending_min
      FROM grid g JOIN j ON j.partition = g.partition
        AND j.eligible_ts <= g.ts AND j.start_ts > g.ts
      GROUP BY 1, 2
    )
    SELECT g.ts, g.partition, g.nodes_total, g.cpus_total,
           COALESCE(r.jobs_running, 0) AS jobs_running,
           COALESCE(r.nodes_alloc, 0) AS nodes_alloc,
           COALESCE(r.cpus_alloc, 0) AS cpus_alloc,
           COALESCE(p2.jobs_pending, 0) AS jobs_pending,
           COALESCE(p2.cpus_pending_requested, 0) AS cpus_pending_requested,
           COALESCE(p2.oldest_pending_min, 0) AS oldest_pending_min
    FROM grid g LEFT JOIN run r USING (ts, partition)
                LEFT JOIN pend p2 USING (ts, partition)
    ORDER BY 1, 2
    """
    s = con.sql(sql).df()
    n = len(s)
    # nodes running work that is not in the accounting export (interactive/OnDemand)
    extra = np.round(s["nodes_total"] * rng.uniform(0.0, 0.05, n)).astype(int)
    s["nodes_alloc"] = np.minimum(s["nodes_alloc"] + extra, s["nodes_total"])
    s["cpus_alloc"] = np.minimum(s["cpus_alloc"] + extra * 128, s["cpus_total"])
    # drained / down nodes, plus the maintenance reservation on `large`
    down = np.round(rng.gamma(1.2, 1.1, n)).astype(int)
    maint = (s["ts"] >= "2026-07-20 20:00") & (s["ts"] < "2026-07-21 02:00") & (s["partition"] == "large")
    s["nodes_down_drain"] = np.minimum(down + np.where(maint, 40, 0), s["nodes_total"])
    s["nodes_alloc"] = np.minimum(s["nodes_alloc"], s["nodes_total"] - s["nodes_down_drain"])
    s["nodes_idle"] = s["nodes_total"] - s["nodes_alloc"] - s["nodes_down_drain"]
    s["reservation_nodes"] = np.where(maint, 40, 0)
    return s


def build_dataset(out_dir="data"):
    """Write jobs.parquet and sched_15m.parquet, once. Returns both paths."""
    os.makedirs(out_dir, exist_ok=True)
    jp = os.path.join(out_dir, "jobs.parquet")
    sp = os.path.join(out_dir, "sched_15m.parquet")
    if os.path.exists(jp) and os.path.exists(sp):
        print("dataset already built")
        return jp, sp
    rng = np.random.default_rng(SEED)
    jobs = build(rng)
    sched = build_sched(jobs, rng)
    jobs.to_parquet(jp, index=False)
    sched.to_parquet(sp, index=False)
    print(f"built {len(jobs):,} job records and {len(sched):,} scheduler samples")
    return jp, sp


In [ ]:
JOBS_PARQUET, SCHED_PARQUET = build_dataset()

### Swapping in real data later

`build_dataset()` only generates when the files are missing, so pointing this workshop
at your own cluster is a file swap, not a rewrite: export with `sacct`, convert with
the companion script `sacct_to_parquet.py`, and place the resulting `jobs.parquet`
and `sched_15m.parquet` in `data/` before running the cell above.

Three things then need your attention, each a lesson from later in the notebook
arriving early: rewrite `DOMAIN_NOTES` (Part 6) for your time zone and window, replace
`CLUSTER_INVENTORY` (Part 3) with your partitions and rates, and stop trusting the
numbers quoted in this prose — they describe the synthetic month, not yours.
`est_start_ts` will be NULL until you start sampling `squeue --start`; the converter's
docstring shows how.

---
# Part 1: One model response

Start with one model request. The model runs on a remote server. Your code sends a
message and reads the reply, giving you direct control over both sides of the exchange.

Watch three details:

- we send a **list of messages**, not a string
- the reply arrives inside `choices[0].message`
- the model keeps no memory between calls, so a conversation means resending the whole
  history each time

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What does Slurm's Planned field measure? Two sentences."}],
    max_tokens=200,
)

display(Markdown(response.choices[0].message.content))

In [ ]:
response.choices[0].message.role

The model can answer from general knowledge; `Planned` is Slurm's own name for time a
job spent queued, and Part 7 is built on it. Now ask a question that requires access
to this cluster's accounting data.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{
        "role": "user",
        "content": "Which project wasted the most core-hours on our cluster last month?",
    }],
    max_tokens=300,
)

display(Markdown(response.choices[0].message.content))

The model cannot know the answer because it cannot see the accounting database.

> A language model is a text function. It cannot reach your scheduler, your filesystem
> or your clock.

---
# Part 2: The accounting data

Leave the model aside for now. Inspect the data before giving the model access to it.

Two words carry a lot of weight here. A **partition** is Slurm's name for a queue: a
named group of nodes with its own size, hardware and limits. An **allocation** is the
set of resources a job was granted, whether or not it used them.

Slurm records one row per job allocation and additional rows for each **step**
inside it: `4180001.batch`, `4180001.extern`, `4180001.0`. Memory high-water marks live
on the steps; the requested resources live on the allocation. This table has already
been rolled up to one row per allocation, with `max_rss_mb` taken as the maximum across
that job's steps.

Array tasks stay separate. Six thousand array tasks are six thousand rows and six
thousand billable jobs, even though the researcher typed one `sbatch`.

In [ ]:
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute(f"CREATE VIEW jobs AS SELECT * FROM '{JOBS_PARQUET}'")

rows, users, accounts, lo, hi = con.sql("""
    SELECT COUNT(*), COUNT(DISTINCT user), COUNT(DISTINCT account),
           MIN(submit_ts), MAX(end_ts)
    FROM jobs
""").fetchone()
print(f"{rows:,} jobs from {users} users in {accounts} projects")
print(f"covering {lo} to {hi} UTC")

In [ ]:
con.sql("""
    SELECT job_id, user, account, partition, state, submit_ts, eligible_ts,
           est_start_ts, start_ts, timelimit_min, elapsed_min, planned_min,
           req_cpus, total_cpu_min
    FROM jobs WHERE partition = 'large' LIMIT 5
""").df()

### The grain

One row per job allocation. Nine columns carry most of the meaning:

| Column | Meaning |
|---|---|
| `submit_ts` | when the user ran `sbatch` |
| `eligible_ts` | when the job first became runnable; later than submit if it was held or waiting on a dependency |
| `est_start_ts` | when the backfill scheduler predicted it would start |
| `start_ts` | when it actually started |
| `timelimit_min` | walltime the user asked for |
| `elapsed_min` | walltime it actually used |
| `req_cpus` | cores allocated, which is what the project is charged for |
| `total_cpu_min` | CPU time actually consumed, summed over every core |
| `planned_min` | Slurm's own count of minutes spent queued |

Two derived quantities drive everything in this notebook:

```
core-hours charged   = req_cpus * elapsed_min / 60
CPU efficiency       = total_cpu_min / (req_cpus * elapsed_min)
```

`total_cpu_min` sums over cores: a 4-core job that keeps all four busy for 10 minutes
records about 40; if one thread did all the work it records about 10, and CPU
efficiency is 25%. A job that asks for 128 cores and runs one thread on one of them is
charged for 128 and uses 1.

The column names follow the `sacct` fields (`Submit`, `Eligible`, `Start`,
`Timelimit`, `Elapsed`, `Planned`, `TotalCPU`, `MaxRSS`), so everything you write
against this table transfers to a real export. We will introduce the remaining columns
when a question needs them.

### Three queries

Start with a total. Every number you produce later should be checkable against it, and
a number that exceeds it means you have double counted somewhere.

In [ ]:
con.sql("""
    SELECT ROUND(SUM(req_cpus * elapsed_min) / 60 / 1e6, 2) AS million_core_hours
    FROM jobs
""").df()

Break it down by day. Check whether the first and last days cover full days before
comparing them with the days in between.

In [ ]:
con.sql("""
    SELECT CAST(submit_ts AS DATE) AS day,
           COUNT(*) AS jobs,
           ROUND(SUM(req_cpus * elapsed_min) / 60 / 1e3, 0) AS kilo_core_hours
    FROM jobs GROUP BY 1 ORDER BY 1
""").df().head(8)

The file starts and ends partway through a UTC date. Keep that boundary in mind;
Part 6 explains how local time changes the daily totals.

Now group by partition. Count jobs as well as core-hours, and add efficiency. Fourteen
million core-hours spread across 67,000 jobs means something different from four million
spread across 157.

In [ ]:
con.sql("""
    SELECT partition,
           COUNT(*) AS jobs,
           ROUND(SUM(req_cpus * elapsed_min) / 60 / 1e6, 2) AS million_core_hours,
           ROUND(100 * SUM(total_cpu_min) / SUM(req_cpus * elapsed_min), 1) AS cpu_eff_pct
    FROM jobs WHERE elapsed_min > 0
    GROUP BY 1 ORDER BY million_core_hours DESC
""").df()

`gpu` sits at 15% CPU efficiency. Before you write to those users, notice that a GPU
job is *supposed* to leave its cores idle. We will come back to that in Part 4; the
column that matters for those jobs is `gpu_util_pct`.

### Charged against used

Application totals tell you who consumed the machine. They do not tell you who
consumed it usefully.

In [ ]:
PALETTE = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9", "#666666"]

top_projects = con.sql("""
    SELECT account,
           ROUND(SUM(req_cpus * elapsed_min) / 60 / 1e3, 0) AS kch_charged,
           ROUND(SUM(total_cpu_min) / 60 / 1e3, 0) AS kch_used,
           ROUND(100 * SUM(total_cpu_min) / SUM(req_cpus * elapsed_min), 1) AS eff_pct
    FROM jobs WHERE elapsed_min > 0
    GROUP BY 1 ORDER BY kch_charged DESC LIMIT 10
""").df()

fig, ax = plt.subplots(figsize=(9, 4.5))
y = np.arange(len(top_projects))
ax.barh(y + 0.2, top_projects["kch_charged"], height=0.38, color=PALETTE[0], label="charged")
ax.barh(y - 0.2, top_projects["kch_used"], height=0.38, color=PALETTE[1], label="CPU time used")
ax.set_yticks(y, top_projects["account"])
ax.invert_yaxis()
ax.set_xlabel("thousand core-hours over four weeks")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

top_projects

The two largest consumers are almost the same size. `vuw03102` was charged 1.77M
core-hours and used 79.6% of them. `uoa04412` was charged 1.73M and used 38.1%,
leaving 1.07M core-hours charged to a project that did not use them.

A ranking by consumption and a ranking by waste are different rankings. Keep this
table. You will hand it to the first agent.

> Sort by the quantity you intend to act on.

---
# Part 3: One tool call, by hand

A tool is a Python function the model can ask your code to run. A useful tool accepts
simple arguments, does one job, and returns text the model can read.

The next five cells show one complete tool call. The model chooses when to request the
tool and how to use the result; your code decides whether to run it.

### Step 1: write the function

The accounting table records what each job asked for. It does not record what the
hardware could have given it, what the partition limits are, or what an hour costs.
That lives in `scontrol show partition`, in `sinfo`, and in whatever spreadsheet your
finance team keeps.

Here it is a dictionary. On your cluster, this function would shell out to `scontrol`
or read your configuration management repository.

The function returns labelled text rather than a dictionary. Clear labels make the
values easier for the model to use correctly.

In [ ]:
CLUSTER_INVENTORY = {
    "debug":   dict(nodes=4,   cpus_per_node=128, mem_gb_per_node=480,  gpus_per_node=0,
                    cpu="AMD EPYC 7713", max_walltime_min=15,    charge_per_core_hour=0.0),
    "large":   dict(nodes=240, cpus_per_node=128, mem_gb_per_node=480,  gpus_per_node=0,
                    cpu="AMD EPYC 7713", max_walltime_min=4320,  charge_per_core_hour=0.02),
    "long":    dict(nodes=40,  cpus_per_node=128, mem_gb_per_node=480,  gpus_per_node=0,
                    cpu="AMD EPYC 7713", max_walltime_min=20160, charge_per_core_hour=0.02),
    "hugemem": dict(nodes=6,   cpus_per_node=128, mem_gb_per_node=4030, gpus_per_node=0,
                    cpu="AMD EPYC 7713", max_walltime_min=10080, charge_per_core_hour=0.06),
    "gpu":     dict(nodes=24,  cpus_per_node=64,  mem_gb_per_node=480,  gpus_per_node=4,
                    cpu="AMD EPYC 7543", max_walltime_min=1440,  charge_per_core_hour=0.02,
                    gpu="NVIDIA A100 40GB", charge_per_gpu_hour=1.20),
}


def partition_info(partition: str) -> str:
    """Hardware, limits and charging rate for one partition. Not in the job table."""
    p = CLUSTER_INVENTORY.get(partition)
    if p is None:
        return (f"Unknown partition '{partition}'. "
                f"Known partitions: {', '.join(CLUSTER_INVENTORY)}.")
    lines = [f"partition: {partition}"]
    lines += [f"{k}: {v}" for k, v in p.items()]
    lines.append(f"total cores: {p['nodes'] * p['cpus_per_node']}")
    lines.append(f"core-hours available per day: {p['nodes'] * p['cpus_per_node'] * 24:,}")
    return "\n".join(lines)


print(partition_info("large"))

Two facts here cannot be derived from the job table. A `large` node has 128 cores and
480 GB, so a job asking for 128 cores and 480 GB has taken a whole node whatever else
it does. And the partition supplies 737,280 core-hours a day, which is the denominator
for any claim that a project "used a lot".

### Step 2: describe the function to the model

The model cannot read the Python function. It sees a JSON object containing the tool's
name, description and parameter schema. Write that interface for a colleague who will
never see the implementation.

In [ ]:
PARTITION_INFO_SPEC = {
    "type": "function",
    "function": {
        "name": "partition_info",
        "description": (
            "Hardware and policy for one Slurm partition: node count, cores and memory "
            "per node, GPU model, maximum walltime, and the charging rate. Use it to "
            "judge whether a job's request was reasonable or what it cost."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "partition": {
                    "type": "string",
                    "description": "Partition name, for example 'large' or 'gpu'.",
                },
            },
            "required": ["partition"],
        },
    },
}

### Step 3: the model asks

The same API call as Part 1 with one extra argument, `tools`. Print both fields of the
reply.

In [ ]:
messages = [{
    "role": "user",
    "content": "A user asked for 128 cores and 480 GB on the large partition. How much of a node is that?",
}]

response = client.chat.completions.create(
    model=MODEL, messages=messages, tools=[PARTITION_INFO_SPEC], max_tokens=500
)
message = response.choices[0].message

print("content:   ", message.content)
print("tool_calls:", message.tool_calls)

`content` is usually empty when the model requests a tool. `tool_calls` contains the
function name and a JSON string of arguments.

Nothing has run yet. The model has sent a request. Your code can execute it, reject it
or ask a person to approve it.

### Step 4: we run it

This cell is the control point. Put argument validation, permissions, rate limits,
audit logging and human approval here, next to the line that calls the function.

In [ ]:
# Append the model's tool request.
messages.append(message)

# Execute the requested function under our control.
call = message.tool_calls[0]
args = json.loads(call.function.arguments)
if call.function.name == "partition_info":
    result = partition_info(**args)

# Append the function result using the request's id.
messages.append({"role": "tool", "tool_call_id": call.id, "content": result})
print(result)

### Step 5: the model answers

Send the full history back, including the tool result. The model can now answer in
prose.

The `tool_call_id` links each result to its request when the model asks for several
tools at once. We resend the full list because the model keeps no conversation state.

In [ ]:
response = client.chat.completions.create(
    model=MODEL, messages=messages, tools=[PARTITION_INFO_SPEC], max_tokens=500
)
display(Markdown(response.choices[0].message.content))

> The model requests a tool. Your code decides whether to run it and returns the
> result. Execution happens in Step 4.

---
# Part 4: Automate the loop

The next function repeats Steps 3, 4 and 5 until the model returns an answer:

```
repeat:
  send the history and the tool specs to the model
  if it asked for no tools, that reply is the answer, so stop
  otherwise run each tool it asked for, append the results, and go round again
```

The loop is about forty lines. It stays unchanged for the rest of the notebook. From
this point on, you will improve the agent by changing its tools and system prompt.

In [ ]:
ToolSpec = dict
Tool = tuple[Callable[..., str], ToolSpec]
Tools = Mapping[str, Tool]


def display_tool_trace(iteration: int, name: str, args: dict, result: str) -> None:
    """Render one complete tool call for a human following along in a notebook."""
    formatted = result if result.lstrip().startswith("|") else f"```text\n{result}\n```"
    display(Markdown(
        f"#### Tool call {iteration}: `{name}`\n\n"
        f"```json\n{json.dumps(args, indent=2)}\n```\n\n"
        f"{formatted}"
    ))


def run_agent(
    question: str,
    tools: Tools | None = None,
    system_prompt: str = "",
    model: str = MODEL,
    max_iters: int = 10,
    verbose: bool = True,
) -> str:
    """Run the agent using a mapping of tool names to (function, spec) pairs."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    available_tools = tools or {}
    specs = [spec for _, spec in available_tools.values()]

    for i in range(max_iters):
        message = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=specs or None,
            max_tokens=2000,
        ).choices[0].message

        if not message.tool_calls:
            if verbose:
                print("\n\n")
                display(Markdown("## Final Answer"))
            return message.content

        messages.append(message)
        for call in message.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            try:
                fn, _ = available_tools[call.function.name]
                result = str(fn(**args))
            except Exception as e:
                result = f"Error: {type(e).__name__}: {e}"

            if verbose:
                display_tool_trace(i + 1, call.function.name, args, result)

            messages.append(
                {"role": "tool", "tool_call_id": call.id, "content": result}
            )

    return (
        f"Stopped after {max_iters} tool iterations without a final answer. "
        "Review the trace, tool errors and prompt before trying again."
    )

### Ten projects, and a bill

Unused core-hours are not free. Somebody paid for the node, the power and the floor
space while a core sat idle. Putting a number on that is what turns an efficiency
report into a conversation a research software engineer can have with a group.

Give the agent the table from Part 2 and let it fetch the rates itself.

`ROLE` below is the **system prompt**: standing instructions sent ahead of the user's
question, the same on every iteration of the loop. Everything we teach the agent from
here on — schema cards, domain notes — will be appended to it.

In [ ]:
ROLE = """You are an HPC platform analyst for a national research computing service.
Answer the question using the tools available. Do not guess at numbers.
Be concise. State the number, the time window it covers, and any caveat that would
change how someone acts on it."""

PARTITION_TOOLS: dict[str, Tool] = {
    "partition_info": (partition_info, PARTITION_INFO_SPEC),
}

USER_MESSAGE = (
    "These are the ten projects charged the most core-hours over the last four weeks. "
    "kch means thousand core-hours.\n\n"
    f"{top_projects.to_markdown(index=False)}\n\n"
    "Almost all of this ran on the large partition. Which projects should I contact "
    "first, and what did their unused core-hours cost at our standard rate? "
    "Check whether the rate is the same everywhere before you apply one number to all "
    "of them, and tell me which partitions you checked."
)

display(Markdown(run_agent(
    question=USER_MESSAGE,
    tools=PARTITION_TOOLS,
    system_prompt=ROLE,
)))

The agent should reach `uoa04412` rather than the largest consumer, and should notice
that `hugemem` and `gpu` are charged differently. Roughly 1.07M unused core-hours at
$0.02 is about $21,000 of machine time in four weeks, from one project.

### Add a second tool

The agent can price the waste, but it still relies on us to supply the table. Package
that query as a tool so it can ask for itself.

The loop stays the same. Only the tool dictionary changes.

In [ ]:
def worst_efficiency_jobs(account: str, n: int = 10) -> str:
    """The jobs in one project that wasted the most core-hours, over the whole month."""
    df = con.sql("""
        SELECT job_name, user, partition, COUNT(*) AS jobs,
               ROUND(SUM(req_cpus * elapsed_min) / 60, 0) AS core_hours_charged,
               ROUND(SUM(req_cpus * elapsed_min - total_cpu_min) / 60, 0) AS core_hours_wasted,
               ROUND(100 * SUM(total_cpu_min) / SUM(req_cpus * elapsed_min), 1) AS cpu_eff_pct
        FROM jobs
        WHERE account = ? AND elapsed_min > 0
        GROUP BY 1, 2, 3
        ORDER BY core_hours_wasted DESC LIMIT ?
    """, params=[account, int(n)]).df()
    if df.empty:
        return f"No jobs found for account '{account}'."
    return df.to_markdown(index=False)


WORST_EFFICIENCY_SPEC = {
    "type": "function",
    "function": {
        "name": "worst_efficiency_jobs",
        "description": (
            "For one project account code, list the job names that wasted the most "
            "core-hours: charged core-hours, wasted core-hours and CPU efficiency."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "account": {"type": "string", "description": "Project account code, e.g. 'uoa04412'."},
                "n": {"type": "integer", "description": "How many rows to return."},
            },
            "required": ["account"],
        },
    },
}

SCOPED_TOOLS: dict[str, Tool] = {
    **PARTITION_TOOLS,
    "worst_efficiency_jobs": (worst_efficiency_jobs, WORST_EFFICIENCY_SPEC),
}

In [ ]:
display(Markdown(run_agent(
    question=("Project uoa04412 is our worst performer. What exactly are they running, "
              "and what should I say to them?"),
    tools=SCOPED_TOOLS,
    system_prompt=ROLE,
)))

### The agent chains the tools

The trace should show two stages: find the jobs, then look up what the hardware could
have given them. The prompt never specified that sequence; the tool descriptions
supplied enough context.

One job name, `kauri_bin_annotate`, accounts for most of it: 6,000 array tasks, each
holding 128 cores to run something that used one. Hold that thought until Part 10.

---
# Part 5: From narrow tools to SQL

The agent can now price waste and inspect one project. Try questions outside those two
jobs:

- Which user has jobs that keep hitting their time limit?
- Are people asking for far more memory than they use?
- How long do jobs wait before they start?

None fits the current tools. Watch what happens.

In [ ]:
display(Markdown(run_agent(
    question="Which user should I contact about jobs that repeatedly hit their time limit?",
    tools=SCOPED_TOOLS,
    system_prompt=ROLE,
)))

> A narrow tool supports only the questions its author anticipated. When no tool fits,
> the model can refuse, guess or misuse one.

In fairness, that was a rigged contest: the question was chosen to defeat two tools
written two cells earlier. It demonstrates the failure mode rather than a fair test.
Part 12 puts the agent against a baseline that can actually win.

### One general tool instead

Replace the question-specific query tool with one function that accepts SQL and returns
rows. It can answer a much wider range of questions, but it can also write inefficient
or misleading queries.

In [ ]:
def run_sql(sql: str) -> str:
    """Run a SQL query and return the rows."""
    return con.sql(sql).df().to_markdown(index=False)


RUN_SQL_SPEC = {
    "type": "function",
    "function": {
        "name": "run_sql",
        "description": "Run a DuckDB SQL query against the Slurm accounting data.",
        "parameters": {
            "type": "object",
            "properties": {"sql": {"type": "string", "description": "A SQL query."}},
            "required": ["sql"],
        },
    },
}

SQL_AGENT_TOOLS: dict[str, Tool] = {
    **PARTITION_TOOLS,
    "run_sql": (run_sql, RUN_SQL_SPEC),
}

SQL replaces `worst_efficiency_jobs`. Keep `partition_info` because it reaches
information the database does not hold.

> A tool earns its place when it does something your general tool cannot.

Ask the time limit question again. Watch the trace rather than judging only the final
answer.

In [ ]:
display(Markdown(run_agent(
    "Which user should I contact about jobs that repeatedly hit their time limit?",
    tools=SQL_AGENT_TOOLS,
    system_prompt=ROLE,
)))

### It has SQL but no context about what it is querying

The agent can inspect the database and eventually find an answer. That costs extra
calls, and introspection cannot tell it that `req_cpus` is what you charge for while
`total_cpu_min` is what was used, or that `elapsed_min` is wall time and
`total_cpu_min` is summed over every core.

Add a **schema card**: a short note that names the table, grain, columns and units.
Write it for a colleague who has thirty seconds to understand the data.

In [ ]:
SCHEMA = """

## Table: jobs
One row per Slurm job allocation, already rolled up from job steps. 87,439 rows
covering four weeks. Array tasks are separate rows. Use it to answer "who ran what,
how long did it wait, and how well did it use what it was given".

Identity:
  job_id           BIGINT
  array_job_id     BIGINT, shared by every task of one array; -1 if not an array
  array_task_id    BIGINT, index within the array; -1 if not an array
  job_name         VARCHAR, the sbatch --job-name
  user             VARCHAR, the submitting username
  account          VARCHAR, the project code the job is charged to
  project_name     VARCHAR, human readable project title
  institution      VARCHAR
  partition        VARCHAR, one of debug, large, long, hugemem, gpu
  qos              VARCHAR, quality of service: a policy bucket that adjusts priority
                   and limits ('debug' jumps the queue, 'normal' does not)
  state            VARCHAR, one of COMPLETED, FAILED, TIMEOUT, CANCELLED,
                   OUT_OF_MEMORY, NODE_FAIL
  exit_code        VARCHAR
  last_reason      VARCHAR, the last pending reason Slurm recorded, NULL if it never
                   waited. Priority: outranked by other jobs. Resources: next in
                   line, waiting for cores to free. Dependency: waiting on another
                   job. ReqNodeNotAvail: nodes held, usually for maintenance

Timestamps, all UTC:
  submit_ts        when sbatch ran
  eligible_ts      when the job became runnable. Later than submit_ts if it was held
                   or waiting on a dependency
  est_start_ts     the backfill scheduler's predicted start, recorded while pending.
                   NULL when Slurm produced no estimate
  start_ts         when it actually started. NULL if it never started
  end_ts           when it stopped for any reason

Requested, and therefore charged:
  timelimit_min    walltime requested
  req_nodes, req_cpus, req_gpus
  req_mem_mb       memory requested, total across the allocation

Used:
  elapsed_min      wall time actually used. NULL if the job never started
  total_cpu_min    CPU time consumed, summed over every allocated core
  max_rss_mb       peak resident memory across the job's steps
  gpu_util_pct     mean GPU utilisation from job profiling, NULL outside the gpu
                   partition
  planned_min      minutes spent queued, measured from eligible_ts to start_ts

Derived quantities, computed rather than stored:
  core-hours charged = req_cpus * elapsed_min / 60
  CPU efficiency     = total_cpu_min / (req_cpus * elapsed_min)
  memory efficiency  = max_rss_mb / req_mem_mb
  walltime efficiency= elapsed_min / timelimit_min
Charging follows the request, not the use. Never average an efficiency ratio across
jobs of different sizes; sum the numerator and the denominator instead.
"""

display(Markdown(run_agent(
    "Are people asking for far more memory than they use? Give me the worst partitions.",
    tools=SQL_AGENT_TOOLS,
    system_prompt=ROLE + SCHEMA,
)))

---
# Part 6: When is the cluster busy?

The schema card describes each column accurately. Ask for submissions by hour.

In [ ]:
display(Markdown(run_agent(
    "When do researchers submit jobs? Give me submissions by hour of day.",
    tools=SQL_AGENT_TOOLS,
    system_prompt=ROLE + SCHEMA,
)))

### Researchers do not mostly work at 11 p.m.

Inspect the SQL before assuming the model made a calculation error.

In [ ]:
con.sql("""
    SELECT EXTRACT(hour FROM submit_ts) AS hour_of_day, COUNT(*) AS jobs
    FROM jobs GROUP BY 1 ORDER BY jobs DESC LIMIT 3
""").df()

`EXTRACT(hour FROM submit_ts)` does peak at 23:00. The calculation is correct, but the
answer uses the wrong time zone. The timestamps are UTC; Kōwhai is in New Zealand,
twelve hours ahead.

The schema card says only `submit_ts  when sbatch ran`, and that the timestamps are
UTC. That is accurate and still not enough: it does not say what a *day* means to the
people who use this machine. Add the sentence that makes the timestamps meaningful.

In [ ]:
DOMAIN_NOTES = """

## Context the tables omit
- All timestamps are UTC. The cluster and its users are in New Zealand, NZST (UTC+12)
  for the whole of this window. Any question about time of day, or about a named day
  such as "Sunday", means local time. Add 12 hours to convert.
- The data covers Monday 6 July to Sunday 2 August 2026 NZST, four complete local
  weeks. In UTC that is 2026-07-05 12:00 to 2026-08-02 12:00.
- New Zealand observes daylight saving from late September to early April. This window
  is entirely NZST, so a fixed +12 is safe here and would not be in October.
"""

display(Markdown(run_agent(
    "When do researchers submit jobs? Give me submissions by hour of day, in local time.",
    tools=SQL_AGENT_TOOLS,
    system_prompt=ROLE + SCHEMA + DOMAIN_NOTES,
)))

The corrected result peaks at 11:00 and bottoms out around 04:00, which is what a
working week looks like.

> An HPC analyst would question an 11 p.m. peak. The agent lacked the sentence that
> makes the timestamps meaningful.

That missing sentence moves every submission twelve hours, which is enough to move
half a day's jobs into the wrong local day, and enough to make a Saturday incident
look as though it began on Friday.

---
# Part 7: Three numbers for one wait

"How long do jobs wait?" sounds like one question. It hides three clocks — when the
job was submitted, when it became eligible, and when the scheduler predicted it would
start — and the answer changes depending on which clock you read and how you average.

Ask the agent, then take the query apart.

In [ ]:
display(Markdown(run_agent(
    "What is the average queue wait on this cluster?",
    tools=SQL_AGENT_TOOLS,
    system_prompt=ROLE + SCHEMA + DOMAIN_NOTES,
)))

### The same jobs, measured four ways

In [ ]:
con.sql("""
    SELECT ROUND(AVG(date_diff('minute', submit_ts, start_ts)) / 60, 2) AS mean_submit_to_start_h,
           ROUND(MEDIAN(date_diff('minute', submit_ts, start_ts)), 0) AS median_submit_to_start_min,
           ROUND(AVG(planned_min) / 60, 2) AS mean_planned_h,
           ROUND(MEDIAN(planned_min), 0) AS median_planned_min
    FROM jobs WHERE start_ts IS NOT NULL
""").df()

Four numbers for the same 86,606 jobs. Reading across: the mean and median of
submit-to-start (4.7 hours, 72 minutes), then the mean and median of Slurm's own
`planned_min` (2.9 hours, 26 minutes). Each is arithmetically correct. Two differences
produce the elevenfold spread.

**Submit is not eligible.** A job that waits on a dependency, or that a user submits
held, is not queued: it is not yet allowed to run. Slurm measures `Planned` from
`eligible_ts`, and counting from `submit_ts` bills the scheduler for the pipeline's
own sequencing.

In [ ]:
con.sql("""
    SELECT CASE WHEN date_diff('minute', submit_ts, eligible_ts) > 2
                THEN 'held or waiting on a dependency'
                ELSE 'eligible at submit' END AS category,
           COUNT(*) AS jobs,
           ROUND(MEDIAN(date_diff('minute', submit_ts, eligible_ts)), 0) AS median_hold_min,
           ROUND(MEDIAN(planned_min), 0) AS median_queue_min
    FROM jobs WHERE start_ts IS NOT NULL GROUP BY 1
""").df()

Nearly 39,000 jobs, the Nextflow and Snakemake tasks, were held a median of 75 minutes
before they were allowed to run. Once eligible, they queued for about as long as
everything else.

**The mean is not the middle.** Queue waits are long-tailed. The mean is a statement
about the worst week of the month; the median is a statement about a typical job.
Report both, or report a percentile.

In [ ]:
con.sql("""
    SELECT ROUND(MEDIAN(planned_min), 0) AS p50_min,
           ROUND(QUANTILE_CONT(planned_min, 0.90), 0) AS p90_min,
           ROUND(QUANTILE_CONT(planned_min, 0.99) / 60, 1) AS p99_h,
           ROUND(MAX(planned_min) / 60, 1) AS max_h
    FROM jobs WHERE start_ts IS NOT NULL
""").df()

And one more trap, which no average will warn you about:

In [ ]:
con.sql("""
    SELECT COUNT(*) AS all_jobs,
           COUNT(start_ts) AS jobs_that_started,
           COUNT(*) - COUNT(start_ts) AS never_started
    FROM jobs
""").df()

833 jobs were cancelled while still pending. They have no `start_ts`, so every average
above silently drops them. If you are asked "how long did people wait", the jobs
somebody gave up on are part of the answer.

### Estimated against real

Slurm schedules in two passes. The main loop starts jobs strictly in priority order.
The **backfill** loop then scans further down the queue for jobs small enough to slip
into gaps without delaying anything above them, and to find those gaps it builds a
timetable of when every running job will finish. That timetable is where the predicted
start time comes from, the one a user sees in `squeue --start`.

Our export keeps each job's prediction in `est_start_ts`. Compare it with what
happened.

In [ ]:
con.sql("""
    SELECT COUNT(*) AS jobs_with_an_estimate,
           ROUND(MEDIAN(date_diff('minute', eligible_ts, est_start_ts)), 0) AS median_estimated_min,
           ROUND(MEDIAN(planned_min), 0) AS median_actual_min,
           ROUND(100.0 * SUM(CASE WHEN start_ts < est_start_ts THEN 1 ELSE 0 END)
                 / COUNT(*), 1) AS pct_started_earlier_than_predicted
    FROM jobs WHERE est_start_ts IS NOT NULL AND start_ts IS NOT NULL
""").df()

The estimate is not noisy. It is biased in one direction: 88% of jobs started earlier
than promised, and the median prediction was roughly three times the median wait.

The cause is in the efficiency data.

In [ ]:
con.sql("""
    SELECT ROUND(MEDIAN(elapsed_min * 1.0 / timelimit_min), 3) AS median_walltime_used,
           ROUND(AVG(elapsed_min * 1.0 / timelimit_min), 3) AS mean_walltime_used
    FROM jobs WHERE state = 'COMPLETED'
""").df()

Backfill plans the future by assuming every running job will use its **full time
limit**, because that is the only promise it has. The median completed job uses 25% of
the limit it asked for. So the schedule the scheduler is reasoning about is roughly
four times longer than the one that actually happens, and every prediction inherits
that.

> The queue estimate is pessimistic because walltime requests are. The efficiency
> problem and the wait-time problem are the same problem seen from two ends.

This is also the honest answer to "should I trust `squeue --start`": treat it as an
upper bound, not a forecast. And the lever for shortening it is not more hardware; it
is users who request the walltime they need.

In [ ]:
DOMAIN_NOTES = DOMAIN_NOTES + """
## How to talk about queue time
- planned_min is the queued time Slurm itself reports, measured from eligible_ts. It
  is the right default for "how long did it wait".
- date_diff('minute', submit_ts, start_ts) also counts time the job was held or
  waiting on a dependency, which is the workflow's own sequencing rather than queue
  pressure. Use it only when the user asks how long from sbatch to running, and say
  which one you used.
- Queue waits are long-tailed. Report the median with p90, never a bare mean.
- Jobs cancelled while pending have start_ts IS NULL and are dropped by every average.
  Count them separately.
- est_start_ts is the backfill scheduler's prediction. It assumes every running job
  runs to its full time limit, so it is systematically pessimistic. Treat a gap
  between est_start_ts and start_ts as expected, not as an anomaly.
"""

display(Markdown(run_agent(
    "What is a typical queue wait on this cluster, and how much should users trust the "
    "start time squeue shows them?",
    tools=SQL_AGENT_TOOLS,
    system_prompt=ROLE + SCHEMA + DOMAIN_NOTES,
)))

### Your turn

Pick one of these and edit the question above:

- Does queue wait depend on job size? Compare median `planned_min` by `req_nodes`
  bucket, and say whether a bigger job waits longer per job or per core-hour.
- Which partition has the worst tail? Rank partitions by p90 rather than by mean.
- Does the estimate get worse when the cluster is busy? Group the estimate ratio by
  day and see which days are worst.

---
# Part 8: Guardrails

The schema card and the time-zone note improved the answers. The SQL tool still accepts
any query and can return any number of rows. Test three failure modes before adding
guardrails.

### Problem 1: a result can flood the context window

This reasonable-looking query returns one row per job.

In [ ]:
per_job = run_sql("""
    SELECT job_id, user, account, partition, state, req_cpus, elapsed_min
    FROM jobs ORDER BY req_cpus * elapsed_min DESC
""")
print(f"{len(per_job):,} characters, roughly {len(per_job) // 4:,} tokens")

The result is larger than many context windows, and every later model call in the loop
would resend it.

### Problem 2: queries can omit the time window

Here the whole dataset is one month in one file. A real Slurm accounting database
(`slurmdbd`) holds years, and a query with no date predicate scans all of it. The habit is worth enforcing
while it is cheap.

### Problem 3: a guessed value returns zero rows and no error

In [ ]:
print("guessed spelling:", len(con.sql(
    "SELECT * FROM jobs WHERE project_name LIKE '%Wananga%'"
).df()), "rows")

con.sql("""
    SELECT account, project_name,
           ROUND(SUM(req_cpus * elapsed_min) / 60 / 1e3, 0) AS kilo_core_hours
    FROM jobs WHERE project_name ILIKE '%Wānanga%'
    GROUP BY 1, 2 ORDER BY kilo_core_hours DESC LIMIT 5
""").df()

Every one of these project names is stored with macrons, and with an em dash rather
than a hyphen between the institution and the topic. A guess without them returns zero
rows even though those projects consumed millions of core-hours. The query succeeds, so
no error draws attention to the mistake.

This is not a New Zealand quirk. Any accounting database has stored strings the user
will not type exactly: `nesi00119`, `Māui`, `p_awi_pool`, `LargeMem_v2`.

### Add guardrails to the tool

The next version caps rows, requires a time filter and adds a lookup tool for stored
values. Read its error messages. Each one tells the model how to correct the failed
request.

This guardrail is designed for the workshop. The prefix check makes the control easy to
see. Production systems need database-level isolation for untrusted SQL: a read-only
role, a statement timeout and a separate replica.

In [ ]:
MAX_ROWS = 50
TIME_COLUMNS = ("submit_ts", "eligible_ts", "start_ts", "end_ts", "ts")

# Read the window from the data, so this guardrail's hint stays correct if you
# swap in a real export.
DATA_LO, DATA_HI = con.sql("SELECT MIN(submit_ts), MAX(end_ts) FROM jobs").fetchone()


def run_sql(sql: str) -> str:
    """Run one workshop-approved SQL query and return the rows as a markdown table."""
    if not re.match(r"\s*(select|with)\b", sql, re.I):
        return "Error: only SELECT queries are allowed. This tool cannot modify data."

    pattern = r"\b(" + "|".join(TIME_COLUMNS) + r")\s*(>=|>|<=|<|between)"
    if not re.search(pattern, sql, re.I):
        return (
            "Error: every query must filter on a timestamp column "
            f"({', '.join(TIME_COLUMNS)}).\n"
            "Add a predicate such as:\n"
            f"  WHERE submit_ts >= TIMESTAMP '{DATA_LO}'\n"
            f"    AND submit_ts <  TIMESTAMP '{DATA_HI}'\n"
            f"The data covers {DATA_LO} to {DATA_HI} UTC. Timestamps are UTC and the "
            "cluster is NZST (UTC+12), so a local day starts at 12:00 UTC the day "
            "before."
        )

    try:
        df = con.sql(sql).df()
    except Exception as e:
        return f"Error: {e}"

    if df.empty:
        return ("0 rows. The query is valid but nothing matched it. Check your filter "
                "values with list_values before concluding the answer is zero.")
    if len(df) > MAX_ROWS:
        return (df.head(MAX_ROWS).to_markdown(index=False)
                + f"\n\n[truncated at {MAX_ROWS} rows. Aggregate, or add ORDER BY and LIMIT]")
    return df.to_markdown(index=False)


LOOKUP_COLUMNS = ("account", "project_name", "institution", "partition", "state",
                  "job_name", "user", "last_reason")


def list_values(column: str, contains: str = "") -> str:
    """Real distinct values of a dimension column, so the agent stops guessing them."""
    if column not in LOOKUP_COLUMNS:
        return f"Error: column must be one of {list(LOOKUP_COLUMNS)}."
    df = con.sql(
        f"SELECT DISTINCT {column} AS value FROM jobs "
        f"WHERE {column} ILIKE ? ORDER BY 1 LIMIT 100",
        params=[f"%{contains}%"],
    ).df()
    if df.empty:
        return f"No {column} value contains '{contains}'. Try a shorter fragment."
    return "\n".join(df["value"].astype(str)) + (
        "\n[first 100 only]" if len(df) == 100 else ""
    )


RUN_SQL_SPEC = {
    "type": "function",
    "function": {
        "name": "run_sql",
        "description": (
            "Run one read-only DuckDB SELECT query against the Slurm accounting data. "
            "Every query must filter on a timestamp column. Results are capped, so "
            "aggregate rather than selecting raw rows."
        ),
        "parameters": {
            "type": "object",
            "properties": {"sql": {"type": "string", "description": "A single SELECT statement."}},
            "required": ["sql"],
        },
    },
}

LIST_VALUES_SPEC = {
    "type": "function",
    "function": {
        "name": "list_values",
        "description": (
            "List the real distinct values of a dimension column before filtering on "
            "it. Use this whenever the user names a project, institution, partition or "
            "job in prose, because the stored spelling and punctuation will not match "
            "what they typed."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "column": {"type": "string", "enum": list(LOOKUP_COLUMNS)},
                "contains": {"type": "string", "description": "Case-insensitive fragment."},
            },
            "required": ["column"],
        },
    },
}

ANALYST_TOOLS: dict[str, Tool] = {
    **PARTITION_TOOLS,
    "run_sql": (run_sql, RUN_SQL_SPEC),
    "list_values": (list_values, LIST_VALUES_SPEC),
}

Ask a question that names a project in prose. Follow each failed attempt and correction
in the trace.

In [ ]:
display(Markdown(run_agent(
    "How efficiently did the Waikato coastal ocean modelling group use their core-hours?",
    tools=ANALYST_TOOLS,
    system_prompt=ROLE + SCHEMA + DOMAIN_NOTES,
)))

The agent reads the tool's error, changes its request and tries again. The loop
contains no special retry branch; the error message provides the instruction at the
moment it is needed.

> Tool errors are part of the prompt. They appear when the model needs them and add no
> tokens to successful calls.

---
# Part 9: The second table

`jobs` has reached its limit. It records what each job asked for and got, but nothing
about the machine around it. It cannot tell you whether a job waited because the
cluster was full, because nodes were drained for maintenance, or because 6,000 other
jobs were ahead of it.

`sched_15m` samples the scheduler every fifteen minutes, the way a Prometheus exporter
scraping `sinfo` and `squeue` would.

In [ ]:
con.execute(f"CREATE VIEW sched_15m AS SELECT * FROM '{SCHED_PARQUET}'")
con.sql("DESCRIBE sched_15m").df()

In [ ]:
con.sql("SELECT * FROM sched_15m WHERE partition = 'large' LIMIT 5").df()

### Your turn: write the schema card

Inspect the schema and sample rows above, then complete `MY_SCHEMA_SCHED`.

Your card should answer four questions:

- What does one row represent?
- Which columns join this table to `jobs`?
- What is the difference between `nodes_idle` and `nodes_total - nodes_alloc`?
- Which column tells you the queue is backing up, and which tells you the machine is
  merely busy?

In [ ]:
MY_SCHEMA_SCHED = """

## Table: sched_15m
One row per ...

  ts            ...
  ...
"""

display(Markdown(run_agent(
    "On Saturday 25 July, how full was the large partition and how many jobs were "
    "waiting? Compare it with the Saturday before.",
    tools=ANALYST_TOOLS,
    system_prompt=ROLE + SCHEMA + MY_SCHEMA_SCHED + DOMAIN_NOTES,
)))

### One version for comparison

Compare this card with yours. Check the grain, the join, the difference between busy
and backed up, and the warning that the two tables do not reconcile exactly. Add any
missing detail to your version before continuing.

In [ ]:
SCHEMA_SCHED = """

## Table: sched_15m
One row per 15-minute sample, per partition: 13,440 rows covering the same four weeks
as jobs. Join to jobs on partition, and on ts against the interval
[start_ts, end_ts) for running work or [eligible_ts, start_ts) for queued work.
Use it to answer "what state was the machine in at that moment".

  ts                       TIMESTAMP, UTC, start of the 15-minute sample
  partition                VARCHAR, joins to jobs.partition
  nodes_total, cpus_total  size of the partition
  nodes_alloc, cpus_alloc  in use at the sample instant
  nodes_down_drain         nodes unavailable: failed, drained, or held by a reservation
  nodes_idle               nodes_total - nodes_alloc - nodes_down_drain. Idle nodes
                           beside a long queue mean jobs that do not fit, not spare
                           capacity
  reservation_nodes        nodes held by a maintenance reservation
  jobs_running             jobs occupying the partition
  jobs_pending             jobs eligible but not yet started. This is the backlog
  cpus_pending_requested   cores those pending jobs are asking for
  oldest_pending_min       age of the oldest eligible pending job, in minutes

The totals do not reconcile exactly with jobs. nodes_alloc includes interactive and
Open OnDemand sessions that the accounting export does not contain, up to about 5% of
nodes. Treat a small discrepancy as expected and a large one as a finding.

Jobs still running when the export was taken are absent from jobs entirely, so the
last days of the window under-count long jobs. Do not read a trend off the final days.
"""

FULL_SYSTEM_PROMPT = ROLE + SCHEMA + SCHEMA_SCHED + DOMAIN_NOTES

display(Markdown(run_agent(
    "On Saturday 25 July, how full was the large partition and how many jobs were "
    "waiting? Compare it with the Saturday before.",
    tools=ANALYST_TOOLS,
    system_prompt=FULL_SYSTEM_PROMPT,
)))

---
# Part 10: Investigate an incident

Your agent now has a guarded query tool, a value lookup, a partition inventory, two
tables, two schema cards and the notes on time zones and queue time.

### Investigate the weekend

Users complained that the cluster was unusable over the weekend of Saturday 25 July.
Ask what happened and let the agent choose its first steps. If it stalls, inspect the
trace and improve the prompt rather than the loop.

In [ ]:
display(Markdown(run_agent(
    "Researchers complained that the cluster was unusable over the weekend of "
    "Saturday 25 July NZST. What happened, and who caused it?",
    tools=ANALYST_TOOLS,
    system_prompt=FULL_SYSTEM_PROMPT,
    max_iters=15,
)))

### Cross-check the finding

Compare the agent's story against the numbers by hand. Check the job count, the shape
of the request, and what the queue did to everyone else.

In [ ]:
con.sql("""
    SELECT job_name, user, account,
           COUNT(*) AS tasks,
           MAX(req_cpus) AS req_cpus_each,
           ROUND(MEDIAN(elapsed_min), 0) AS median_elapsed_min,
           ROUND(SUM(req_cpus * elapsed_min) / 60 / 1e3, 0) AS kilo_core_hours,
           ROUND(100 * SUM(total_cpu_min) / SUM(req_cpus * elapsed_min), 2) AS cpu_eff_pct
    FROM jobs
    WHERE start_ts >= TIMESTAMP '2026-07-24 12:00'
      AND start_ts <  TIMESTAMP '2026-07-27 12:00'
    GROUP BY 1, 2, 3 ORDER BY kilo_core_hours DESC LIMIT 5
""").df()

In [ ]:
con.sql("""
    SELECT CAST(eligible_ts + INTERVAL 12 HOUR AS DATE) AS local_day,
           COUNT(*) AS jobs,
           ROUND(MEDIAN(planned_min), 0) AS median_queue_min,
           ROUND(QUANTILE_CONT(planned_min, 0.90) / 60, 1) AS p90_queue_h
    FROM jobs
    WHERE partition = 'large' AND start_ts IS NOT NULL
      AND job_name <> 'kauri_bin_annotate'
      AND eligible_ts >= TIMESTAMP '2026-07-20 12:00'
      AND eligible_ts <  TIMESTAMP '2026-07-31 12:00'
    GROUP BY 1 ORDER BY 1
""").df()

One array of 6,000 tasks held 128 cores each to run something that used the equivalent
of one core: 821,000 core-hours charged, 0.86% of them used. That is 4.3% of the
entire month's consumption, spent in two and a half days.

Everyone else's median wait on `large` went from about 25 minutes to about 80 minutes
and stayed there until Monday. The second query excludes the array itself, so that
number is what the incident did to *other people*.

> One project's efficiency is another project's queue time. On a shared machine these
> are the same metric.

Notice also what the state column says: every one of those tasks is `COMPLETED`. There
is no failure here, no alert, no error log. The only signal is the ratio between what
was asked for and what was used.

### Write the workflow you wish had been running on Saturday morning

A workflow is a plain-text procedure you give the agent in place of a single question.
Write it like a runbook for a new person on the service desk: ordered steps, decision
points and the checks required before you contact a research group.

In [ ]:
TRIAGE_WORKFLOW = """
Run the weekend efficiency triage for this cluster, for Saturday 25 July NZST.

1. Find the ten job names that were charged the most core-hours that day, with their
   account, task count, cores per task and CPU efficiency.
2. For the worst one, show core-hours charged per hour of that day so we can see
   whether it ramped or arrived all at once.
3. Establish a baseline: what did that account run in the two weeks before? A group
   that has always been inefficient is a different conversation from one that changed
   overnight.
4. Quantify the effect on everyone else: median and p90 queue wait on that partition,
   that day, excluding the offending jobs, compared with the same weekday earlier in
   the month.
5. Check sched_15m: was the partition actually full, or were nodes idle beside a long
   queue?

Report what happened, when it started, how confident you are, which numbers you would
not put in front of the research group without checking, and the single next thing a
human should do.
"""

display(Markdown(run_agent(
    TRIAGE_WORKFLOW,
    tools=ANALYST_TOOLS,
    system_prompt=FULL_SYSTEM_PROMPT,
    max_iters=25,
)))

### Now write your own

Choose a task your team performs by hand each week. Write it in `MY_WORKFLOW`, then
uncomment the agent call. If you need a starting point, use one of these:

- **The quiet one.** A user resubmitted `seismic_inv_full` 38 times. Every one hit the
  three-day time limit and produced nothing. The tell is `state` and repetition, not
  volume. What would you say to them, and what would you change in the scheduler?
- **A memory audit.** Which projects request the most memory they never touch? Note
  that `OUT_OF_MEMORY` jobs show near-100% memory efficiency, so high efficiency is not
  automatically good news. Make the agent report both, then commit to a recommendation.
- **A GPU report.** The `gpu` partition sits at 15% CPU efficiency by design. Rank GPU
  users by `gpu_util_pct` instead, and state explicitly which metric you used and why
  the obvious one is wrong.
- **A walltime campaign.** If the median job uses 25% of its time limit, which projects
  would most improve everyone's queue estimates by tightening their requests? Estimate
  the effect.

In [ ]:
MY_WORKFLOW = """
...
"""

# display(Markdown(run_agent(
#     MY_WORKFLOW,
#     tools=ANALYST_TOOLS,
#     system_prompt=FULL_SYSTEM_PROMPT,
#     max_iters=20,
# )))

---
# Part 11: A job worth handing over

Every question so far has had a known answer. I wrote the SQL before I wrote the
commentary, and you could have written it too. That makes for a clear workshop and a
weak argument: a demo where the author already knows the answer cannot show you
whether the agent reasoned or simply pattern-matched to the obvious query.

So here is a task with the two properties that actually justify the machinery:

- **volume** — the same work repeated across many subjects
- **prose output** — the deliverable is writing, not a number

Eighteen research groups need a monthly note about their own usage, in language that
means something to a biologist. The SQL behind all eighteen is identical. The writing
is not, and the writing is the part nobody on your team has time for.

In [ ]:
ADVISORY = """Write a short usage note for the research group behind project {account},
covering the four weeks in this dataset.

1. Get their totals: core-hours charged, CPU time actually used, and overall CPU
   efficiency. Remember to filter on a timestamp column.
2. Find their single worst job name by wasted core-hours, and how it was configured:
   cores requested per task, how many tasks, and how long each ran.
3. Look up what a node in that partition actually provides, so you can say what
   fraction of one they were holding.

Then write at most 150 words addressed to the group. No SQL, no column names, no
jargon: they know their science, not Slurm. Say what they were charged, what they
used, name the job, say concretely what to change in their sbatch script, and estimate
what the change would save. Be direct and not preachy; these are colleagues, not
offenders."""

notes = {}
for account in ["uoa04412", "vuw03102", "uoc02640"]:
    notes[account] = run_agent(
        ADVISORY.format(account=account),
        tools=ANALYST_TOOLS,
        system_prompt=FULL_SYSTEM_PROMPT,
        max_iters=12,
        verbose=False,          # the traces would be three times as long as the notes
    )

for account, note in notes.items():
    display(Markdown(f"### {account}\n\n{note}"))

Three notes, one loop, and the same loop runs for all eighteen projects, or for two
hundred. Notice what was actually delegated: the queries are near-identical across the
three, so the agent did no clever analysis. It did **translation at volume** — turning
`total_cpu_min / (req_cpus * elapsed_min)` into a sentence a marine biologist will act
on, eighteen times, in eighteen slightly different situations.

That is a real job, and it is the first one in this notebook that a human would
genuinely refuse to do by hand.

Two caveats worth stating out loud. The agent **drafts**; a person reads and sends,
because a wrong number in a note to a research group costs more trust than the note
was worth. And translation is where this failure mode lives: the arithmetic can be
right while the sentence overstates it. Read one note against its own trace before you
believe the other seventeen.

> Hand over work that is repetitive, low-stakes per item, and needs prose.
> Keep the work where one wrong number matters.

---
# Part 12: Measure it before you trust it

You now have a working agent, which is the point at which it becomes tempting to
deploy one. Three measurements first. None of them takes long, and each answers a
question your colleagues will ask.

### Measurement 1: does it give the same answer twice?

The same question, five times. Watch both the answers and the paths taken to reach
them.

In [ ]:
SEEN_SQL: list[str] = []


def recording_run_sql(sql: str) -> str:
    """Identical to run_sql, but keeps a copy of every query the agent writes."""
    SEEN_SQL.append(" ".join(sql.split()))
    return run_sql(sql)


RECORDING_TOOLS: dict[str, Tool] = {
    **ANALYST_TOOLS,
    "run_sql": (recording_run_sql, RUN_SQL_SPEC),
}

QUESTION = ("How many core-hours did project uoa04412 waste over the four weeks in "
            "this dataset? Reply with the number, then one sentence.")

answers = [
    run_agent(QUESTION, tools=RECORDING_TOOLS, system_prompt=FULL_SYSTEM_PROMPT,
              max_iters=10, verbose=False)
    for _ in range(5)
]

for i, answer in enumerate(answers, 1):
    print(f"--- run {i} ---\n{(answer or '').strip()[:260]}\n")

print(f"{len(SEEN_SQL)} SQL calls in total, {len(set(SEEN_SQL))} of them distinct")

The number should be stable, because the data is unambiguous and the schema card is
good. The *wording* will differ every time, and so, usually, will the SQL: five runs
rarely produce one distinct query.

That matters in two directions. A number that moves between runs means the question is
underspecified, the context is missing something, or both — treat run-to-run variance
as a diagnostic for your prompt rather than a quirk of the model. And prose that moves
between runs means you cannot promise a researcher the same answer twice, which rules
this out of anything that has to be consistent: quota decisions, allocation reviews,
anything a user could appeal.

### Measurement 2: what does an answer cost?

Compare the agent against the query it eventually wrote.

In [ ]:
import time

usage = {"calls": 0, "prompt_chars": 0}
_real_create = client.chat.completions.create


def counting_create(**kwargs):
    usage["calls"] += 1
    usage["prompt_chars"] += sum(len(str(m)) for m in kwargs.get("messages", []))
    return _real_create(**kwargs)


client.chat.completions.create = counting_create
started = time.perf_counter()
_ = run_agent(QUESTION, tools=ANALYST_TOOLS, system_prompt=FULL_SYSTEM_PROMPT,
              max_iters=10, verbose=False)
agent_seconds = time.perf_counter() - started
client.chat.completions.create = _real_create   # always put it back

started = time.perf_counter()
con.sql("""
    SELECT ROUND(SUM(req_cpus * elapsed_min - total_cpu_min) / 60, 0) AS core_hours_wasted
    FROM jobs
    WHERE account = 'uoa04412' AND elapsed_min > 0
      AND submit_ts >= TIMESTAMP '2026-07-05 12:00'
""").df()
sql_seconds = time.perf_counter() - started

print(f"agent : {agent_seconds:6.1f} s, {usage['calls']} model calls, "
      f"~{usage['prompt_chars'] // 4:,} prompt tokens (rough)")
print(f"SQL   : {sql_seconds * 1000:6.1f} ms, 0 model calls, 0 tokens")
print(f"ratio : {agent_seconds / max(sql_seconds, 1e-6):,.0f}x slower")

Four orders of magnitude, for an answer the SQL gives exactly and the agent gives
approximately. Keep that number in mind whenever someone proposes putting an agent in
front of a dashboard.

Note where the tokens go. The loop resends the entire history on every iteration, so
the schema card and domain notes are paid for once per tool call, not once per
question. A six-call investigation pays for its context six times, and cost grows
faster than the number of steps. Long system prompts are cheap to write and expensive
to run.

### Measurement 3: the error that does not announce itself

Every mistake in this notebook so far has been visibly absurd — an 11 p.m. peak, a
query returning zero rows. Those are the safe failures. Here is the dangerous kind.

In [ ]:
con.sql("""
    SELECT ROUND(100 * AVG(total_cpu_min / (req_cpus * elapsed_min)), 1) AS mean_of_ratios,
           ROUND(100 * SUM(total_cpu_min) / SUM(req_cpus * elapsed_min), 1) AS ratio_of_sums
    FROM jobs
    WHERE state = 'COMPLETED' AND elapsed_min > 0
      AND submit_ts >= TIMESTAMP '2026-07-05 12:00'
""").df()

"Cluster CPU efficiency" is 49.7% or 70.3% depending on which of these you write. Both
are plausible. Both would survive a management meeting. Only the second is right: the
first averages a ratio, so a two-minute test job counts as much as a three-day
64-node run.

Nothing flags this. The query succeeds, the number looks sensible, and no guardrail
can catch it because it is not a malformed query — it is the wrong question, correctly
executed. The schema card warns against it in one line, which is exactly the kind of
protection that works most of the time and fails silently the rest.

> The failures you can see are not the ones to worry about. Budget review time for
> answers that look fine.

### So when should you use one?

Four questions. If a task answers "yes" to the first or fourth, an agent is the
expensive way to do something simpler.

| Ask | If yes |
|---|---|
| Is the answer already known, or asked the same way each time? | Save the query. Put it on a dashboard. |
| Would a threshold on a single ratio catch it? | Write the alert. It fires at 2 a.m.; an agent waits to be asked. |
| Does the output trigger a consequence — a charge, a quota, an email that sends itself? | Keep a human between the answer and the action. |
| Is the work repetitive, low-stakes per item, and does it need prose? | This is the job. Hand it over. |

Scored honestly, most of this notebook fails its own test:

| Question from this notebook | What it should actually be |
|---|---|
| Which project wasted the most core-hours? (Parts 2, 4) | a saved query on a dashboard |
| Who repeatedly hits the time limit? (Part 5) | a saved query, reviewed monthly |
| When do people submit? (Part 6) | a saved query; the answer changes slowly |
| What is a typical queue wait? (Part 7) | a saved query, plus one page written once by a human |
| The weekend incident (Part 10) | a threshold alert on requested-versus-used cores |
| Eighteen tailored advisory notes (Part 11) | an agent |
| "Something is wrong and I do not know what yet" | an agent, as a first pass a human checks |

That is not an argument against building this. It is the argument for knowing which
row you are in before you start. The parts that fail the test still taught you the
schema card, the domain notes and the guardrails — and those are the durable asset.
They make your data legible to a colleague, a dashboard, a new hire and an agent
alike, and they outlive whichever model you are calling this month.

> The agent is the cheapest part of this to build and the least valuable thing you
> made. The context is the asset.

### Your turn: score a real task

Take something your team does by hand each week and put it through the four questions.
Be strict with the first and second: most reporting work is a saved query wearing a
costume.

In [ ]:
MY_DECISION = """
Task:

1. Is the answer already known, or asked the same way each time?
2. Would a threshold catch it?
3. Does the output trigger a consequence?
4. Is it repetitive, low-stakes per item, and does it need prose?

Verdict:
What I will build instead, if the answer is no:
"""

print(MY_DECISION)

---
## What to take home

| Principle | Practical consequence |
|---|---|
| The agent loop is small | Most capability comes from tools and context |
| Your code executes tools | Validation and approval belong at the function call |
| Keep tools distinct | Add a tool when the general SQL tool cannot do the job |
| Schema and meaning differ | Document both the column structure and the operational context |
| Tool errors guide retries | Write errors that tell the model how to recover |
| Enforce limits in code | Row caps are controls; prompt requests are suggestions |
| Charging follows the request | Efficiency is a ratio between two columns, not a column |
| A correct query can still answer the wrong question | Submit, eligible and start are three different clocks |
| Success is not efficiency | The worst incident in this dataset is 6,000 COMPLETED jobs |
| Most questions do not need an agent | A saved query is faster, cheaper and gives the same answer twice |
| Agents earn their place on volume and prose | Eighteen tailored notes, not one number |
| Measure before deploying | Same question, five runs, five different paths |
| The context outlives the model | Schema cards and domain notes serve dashboards and colleagues too |

An answer is limited by the data the agent can access and the context you provide.
State those limits when someone may act on the result.

## Mapping this to a real cluster

Both tables here are stand-ins. On a Slurm cluster you already have the sources.

**`jobs`** comes from `sacct`:

```
sacct -a -X -S 2026-07-06 -E 2026-08-03 --parsable2 --noconvert \
  --format=JobID,JobIDRaw,JobName,User,Account,Partition,QOS,State,ExitCode,\
Submit,Eligible,Start,End,Timelimit,Elapsed,Planned,NNodes,NCPUS,ReqMem,ReqTRES,\
AllocTRES,TotalCPU,Reason
```

Three things to know before you trust the output:

- `-X` returns allocations only. It is what you want for one row per job, but `MaxRSS`
  is a **step** field and comes back empty. For memory you need the steps, which means
  dropping `-X` and rolling up, or using `seff` per job, or reading your profiling
  data from the job profile plugin.
- `Planned` was called `Reserved` before Slurm 23.02. On older clusters, compute it as
  `Start - Eligible`.
- `--noconvert` keeps memory in bytes rather than the mixed `Gn`/`Mc` suffixes, which
  are per-node or per-core depending on how the job was submitted.

**`sched_15m`** comes from sampling. `sinfo -O NodeHost,StateLong` and
`squeue -O JobID,State,Reason,NumCPUs` on a timer, or the node and job state series
your Prometheus Slurm exporter already collects. `sdiag` adds the scheduler's own view:
backfill cycle time, queue depth, and how many jobs the last backfill pass could see.

**`est_start_ts`** is not retained by Slurm. `squeue --start` shows the current
prediction for pending jobs and nothing keeps it afterwards, so if you want to compare
predicted with actual you have to record it yourself: sample `squeue --start` on a
timer and store the first estimate each job receives.

**`partition_info`** is `scontrol show partition` plus your charging policy.

The one thing you cannot get from any command is `DOMAIN_NOTES`. That is the part your
team knows and the machine does not.

## From notebook to package

This notebook is a teaching artifact, and most of it should not be shipped. Of its
56 code cells, 11 contain reusable definitions and the rest are demonstrations —
including several that are deliberately wrong. Extracting it mechanically would
carry the 11 p.m. peak and the 49.7% efficiency figure into production as if they
were features.

The companion package `kowhai-agent` is the other half: the same loop, the same
three tools and the same guardrails, with three changes that only matter once
something runs unattended.

| Here | There | Why |
|---|---|---|
| `SCHEMA`, `DOMAIN_NOTES` as Python strings | markdown files in `context/` | content your colleagues can edit and review |
| hand-written JSON tool specs | `@tool` generates them from the signature | the spec cannot drift from the function |
| cost measured by hand in Part 12 | `Run.summary()` and `logs/runs.jsonl` | an agent you cannot cost is one you cannot defend |

```bash
pip install -e ./kowhai-agent
kowhai selfcheck                          # every tool, no model calls
kowhai advisory --out drafts/             # Part 11, as a scheduled job
```

Keep this notebook standalone so it still runs in Colab with nothing installed. The
package is where the work goes once you have decided, using Part 12's rubric, that
it is work worth automating at all.

## Self-check

Runs every tool without calling a model. If a cell above failed, run this to find out
whether the problem is in the setup and tools or in the model call.

In [ ]:
def _selfcheck() -> None:
    checks: list[tuple[str, bool, str]] = []

    def check(name, fn):
        try:
            out = str(fn())
            ok = not out.lower().startswith("error")
            checks.append((name, ok, out[:80].replace("\n", " ")))
        except Exception as e:
            checks.append((name, False, f"{type(e).__name__}: {e}"))

    check("jobs view", lambda: con.sql("SELECT COUNT(*) FROM jobs").fetchone()[0])
    check("sched view", lambda: con.sql("SELECT COUNT(*) FROM sched_15m").fetchone()[0])
    check("partition_info", lambda: partition_info("large"))
    check("partition_info bad input", lambda: "ok" if partition_info("nope").startswith("Unknown") else "error")
    check("worst_efficiency_jobs", lambda: worst_efficiency_jobs("uoa04412", 3))
    check("run_sql guarded ok", lambda: run_sql(
        "SELECT COUNT(*) AS n FROM jobs WHERE submit_ts >= TIMESTAMP '2026-07-06'"))
    check("run_sql rejects no-time", lambda: "ok" if run_sql(
        "SELECT COUNT(*) FROM jobs").startswith("Error: every query") else "error")
    check("run_sql rejects write", lambda: "ok" if run_sql(
        "DROP VIEW jobs").startswith("Error: only SELECT") else "error")
    check("list_values", lambda: list_values("account", "uoa"))
    check("list_values bad column", lambda: "ok" if list_values("secret").startswith("Error") else "error")
    check("api key present", lambda: "ok" if client.api_key else "error")

    width = max(len(n) for n, _, _ in checks)
    for name, ok, detail in checks:
        print(f"{'PASS' if ok else 'FAIL'}  {name:<{width}}  {detail}")


_selfcheck()